# Homework 1.2: The Flat-Earth Nonholonomic Kepler Problem

## Mathematical Foundations of Generative AI

## Project Goal

The goal of this project was to use a large language model (LLM) as a mechanics assistant for a simplified nonholonomic Kepler problem. The model assumes that a flat disk rolls without slipping on a plane while being attracted toward a fixed Sun at the origin by an inverse-square gravitational force.

The LLM was used to help derive the reduced equations of motion, identify quantities that might be conserved, and suggest numerical checks. The equations and predictions were then checked mathematically and tested with numerical simulations rather than accepted automatically.

The main questions were:

1. What are the reduced equations of motion for the rolling disk?
2. What kinds of trajectories appear for different initial conditions?
3. Is total energy conserved?
4. Is ordinary orbital angular momentum conserved?
5. Is heading angular momentum conserved?
6. How does the nonholonomic rolling constraint change the usual Kepler problem?

---

# 1. Model and Reduced Equations of Motion

The center of the disk is

$$
q=(x,y),
$$

and the heading direction is

$$
e_\theta=(\cos\theta,\sin\theta).
$$

The no-slipping constraint requires the center velocity to remain parallel to the heading direction:

$$
\dot q = v e_\theta.
$$

Therefore,

$$
\dot x=v\cos\theta
$$

and

$$
\dot y=v\sin\theta.
$$

Equivalently, the forbidden sideways velocity satisfies

$$
-\dot x\sin\theta+\dot y\cos\theta=0.
$$

This is a velocity constraint rather than a constraint that fixes the position of the disk to a curve.

## Gravitational Potential and Force

The gravitational potential is

$$
U(q)=-\frac{k}{|q|},
$$

where

$$
|q|=\sqrt{x^2+y^2}.
$$

The gravitational force is

$$
F(q)=-\nabla U(q)
$$

and therefore

$$
F(q)
=
-\frac{kq}{|q|^3}
=
-\frac{k(x,y)}{(x^2+y^2)^{3/2}}.
$$

The force components are

$$
F_x=-\frac{kx}{(x^2+y^2)^{3/2}}
$$

and

$$
F_y=-\frac{ky}{(x^2+y^2)^{3/2}}.
$$

Only the component of the gravitational force along the allowed heading direction changes the scalar speed $v$. The projection is

$$
F(q)\cdot e_\theta
=
-\frac{k(x\cos\theta+y\sin\theta)}{(x^2+y^2)^{3/2}}.
$$

The reduced speed equation is

$$
M\dot v=F(q)\cdot e_\theta,
$$

so

$$
\dot v
=
-\frac{k(x\cos\theta+y\sin\theta)}{M(x^2+y^2)^{3/2}}.
$$

## Effective Mass and Heading Inertia

For a homogeneous thin disk,

$$
I_3=\frac{1}{2}mR^2
$$

and

$$
J=\frac{1}{4}mR^2.
$$

The effective translational mass is

$$
M=m+\frac{I_3}{R^2}.
$$

Substituting the thin-disk value of $I_3$ gives

$$
M=m+\frac{1}{2}m=\frac{3}{2}m.
$$

The simplified heading equation is

$$
J\ddot\theta=0.
$$

Defining

$$
\omega=\dot\theta,
$$

we obtain

$$
\dot\theta=\omega
$$

and

$$
\dot\omega=0.
$$

Thus the heading angular velocity is constant.

## Final First-Order System

The state vector is

$$
(x,y,v,\theta,\omega).
$$

The first-order system used in the numerical simulation is

$$
\dot x=v\cos\theta,
$$

$$
\dot y=v\sin\theta,
$$

$$
\dot v
=
-\frac{k}{M}
\frac{x\cos\theta+y\sin\theta}{(x^2+y^2)^{3/2}},
$$

$$
\dot\theta=\omega,
$$

$$
\dot\omega=0.
$$

## Mathematical and Dimensional Checks

The position equations have units of velocity because $v$ has units of length per time while sine and cosine are dimensionless.

From

$$
U=-\frac{k}{r},
$$

$k$ must have dimensions

$$
[k]=ML^3T^{-2}.
$$

Then

$$
\frac{kq}{|q|^3}
$$

has dimensions $MLT^{-2}$, which are the dimensions of force.

Also,

$$
\frac{I_3}{R^2}
$$

has dimensions of mass, so the definition of $M$ is dimensionally consistent.

Finally, $J\ddot\theta$ has dimensions of torque. Thus the reduced equations are dimensionally consistent.

The derivations are contained in:

- `code/derivation(step1).ipynb`
- `code/derivation(step2).ipynb`

---

# 2. Numerical Simulation

The system was simulated with Python using `scipy.integrate.solve_ivp`.

The parameter values were

$$
m=1,
$$

$$
R=1,
$$

$$
k=1.
$$

For the homogeneous thin disk,

$$
I_3=0.5,
$$

$$
J=0.25,
$$

and

$$
M=1.5.
$$

The simulation interval was

$$
0\le t\le100,
$$

with 10,000 output times. The numerical solver used tolerances

$$
\text{rtol}=10^{-10}
$$

and

$$
\text{atol}=10^{-12}.
$$

A safety event stopped a simulation if the disk came within radius $0.1$ of the singularity at the Sun. None of the four simulations reached this cutoff.

The four initial conditions were:

| Case | $x_0$ | $y_0$ | $v_0$ | $\theta_0$ | $\omega_0$ |
| --- | ---: | ---: | ---: | ---: | ---: |
| Case 1 | 2.0 | 0.0 | 0.8 | $\pi/2$ | 0.20 |
| Case 2 | 2.0 | 0.0 | 0.8 | $\pi/2$ | 0.40 |
| Case 3 | 2.0 | 0.0 | 1.0 | $\pi/2$ | 0.20 |
| Case 4 | 2.0 | 0.0 | 0.8 | $\pi/3$ | 0.20 |

The complete simulation and conservation-check code is stored in:

`code/(step3-7).ipynb`

The numerical time series are stored in:

- `data/case_1_simulation.csv`
- `data/case_2_simulation.csv`
- `data/case_3_simulation.csv`
- `data/case_4_simulation.csv`

## Core Python System

The main ODE function used in the simulation was:

```python
def flat_earth_kepler(t, state):
    x, y, v, theta, omega = state

    r = np.sqrt(x**2 + y**2)

    dxdt = v * np.cos(theta)
    dydt = v * np.sin(theta)

    q_dot_e = x * np.cos(theta) + y * np.sin(theta)

    dvdt = -(k / M) * q_dot_e / r**3

    dthetadt = omega
    domegadt = 0.0

    return [dxdt, dydt, dvdt, dthetadt, domegadt]
```

---

# 3. Trajectories for Several Initial Conditions

The combined trajectory plot is shown below.

![Several initial conditions](result%20images/flat_earth_multiple_trajectories.png)

The trajectories are very different from ordinary Kepler ellipses. The disk cannot accelerate freely in every direction because its center velocity must remain aligned with its heading. Changing the initial speed, initial heading, or heading angular velocity therefore changes the geometry strongly.

## Case 1

![Case 1 trajectory](result%20images/case_1_trajectory.png)

Case 1 produces a bounded looping trajectory over the simulated interval. The path does not close after one revolution and does not resemble an ordinary Kepler ellipse.

## Case 2

![Case 2 trajectory](result%20images/case_2_trajectory.png)

Case 2 produces an almost perfect circle of radius 2. The initial conditions satisfy

$$
\frac{v_0}{r_0}=\frac{0.8}{2}=0.4=\omega_0.
$$

The initial heading is tangent to the circle because $\theta_0=\pi/2$. Along this circular solution,

$$
q\cdot e_\theta=0,
$$

so

$$
\dot v=0.
$$

The speed remains constant while the heading rotates at exactly the rate required to move around a circle of radius 2.

## Case 3

![Case 3 trajectory](result%20images/case_3_trajectory.png)

Case 3 uses the larger initial speed $v_0=1.0$. The resulting loops extend much farther from the Sun than in Case 1. This shows that the spatial scale of the motion is sensitive to the initial translational speed.

## Case 4

![Case 4 trajectory](result%20images/case_4_trajectory.png)

Case 4 changes the initial heading from $\pi/2$ to $\pi/3$. The resulting trajectory is visibly different from Case 1, showing that the heading direction is a dynamically important part of the nonholonomic model.

These plots demonstrate that the model can generate circular and quasi-periodic-looking looping trajectories depending on the initial conditions.

---

# 4. Energy Conservation

The total mechanical energy is

$$
E
=
\frac{1}{2}Mv^2
+
\frac{1}{2}J\omega^2
-
\frac{k}{|q|}.
$$

Since

$$
|q|=\sqrt{x^2+y^2},
$$

the numerical energy was computed as

$$
E
=
\frac{1}{2}Mv^2
+
\frac{1}{2}J\omega^2
-
\frac{k}{\sqrt{x^2+y^2}}.
$$

## Analytical Check

Differentiate the translational kinetic energy:

$$
\frac{d}{dt}\left(\frac{1}{2}Mv^2\right)
=Mv\dot v.
$$

The speed equation gives

$$
M\dot v
=
-\frac{k}{|q|^3}(q\cdot e_\theta).
$$

Since

$$
\dot q=ve_\theta,
$$

we have

$$
q\cdot\dot q=v(q\cdot e_\theta).
$$

The derivative of the potential energy is

$$
\frac{d}{dt}\left(-\frac{k}{|q|}\right)
=
\frac{k}{|q|^3}(q\cdot\dot q)
=
\frac{kv}{|q|^3}(q\cdot e_\theta).
$$

This exactly cancels the derivative of the translational kinetic energy.

Also,

$$
\dot\omega=0,
$$

so

$$
\frac{d}{dt}\left(\frac{1}{2}J\omega^2\right)=0.
$$

Therefore,

$$
\frac{dE}{dt}=0.
$$

The analytical model predicts that total energy is conserved.

## Numerical Results

| Case | Initial Energy | Maximum Absolute Energy Error |
| --- | ---: | ---: |
| Case 1 | -0.015000 | 4.96e-10 |
| Case 2 | approximately 0 | 4.30e-11 |
| Case 3 | 0.255000 | 9.52e-10 |
| Case 4 | -0.015000 | 7.90e-10 |

![Energy conservation](result%20images/energy_conservation_all_cases.png)

To see the small numerical errors more clearly, the quantity

$$
E(t)-E(0)
$$

was plotted.

![Energy error](result%20images/energy_error_all_cases.png)

The energy error remains at approximately $10^{-9}$ or smaller for every simulation. These variations are consistent with floating-point and numerical integration error.

Case 2 deserves special attention. Its theoretical initial energy is exactly zero:

$$
E(0)
=
\frac{1}{2}(1.5)(0.8)^2
+
\frac{1}{2}(0.25)(0.4)^2
-
\frac{1}{2}
=
0.48+0.02-0.50
=0.
$$

Python returned approximately $1.11\times10^{-16}$ because of floating-point roundoff. Therefore, a relative error obtained by dividing by the initial energy is meaningless for Case 2. The absolute energy error is the appropriate quantity.

The numerical simulation confirms that energy is conserved in this simplified nonholonomic model.

---

# 5. Ordinary Orbital Angular Momentum

The ordinary orbital angular momentum about the Sun is

$$
L_{\mathrm{orb}}
=
M(x\dot y-y\dot x).
$$

Using

$$
\dot x=v\cos\theta
$$

and

$$
\dot y=v\sin\theta,
$$

this becomes

$$
L_{\mathrm{orb}}
=
Mv(x\sin\theta-y\cos\theta).
$$

## Numerical Results

| Case | Initial $L_{\mathrm{orb}}$ | Minimum | Maximum | Maximum Relative Change |
| --- | ---: | ---: | ---: | ---: |
| Case 1 | 2.4000 | 2.3995 | 3.4064 | 41.9% |
| Case 2 | 2.4000 | 2.4000 | 2.4000 | approximately 0% |
| Case 3 | 3.0000 | 3.0000 | 6.7778 | 125.9% |
| Case 4 | 2.0785 | 2.0754 | 3.5451 | 70.6% |

![Orbital angular momentum](result%20images/orbital_angular_momentum_all_cases.png)

The change from the initial value is shown below.

![Change in orbital angular momentum](result%20images/orbital_angular_momentum_change.png)

Cases 1, 3, and 4 show large periodic changes in orbital angular momentum. Therefore, ordinary orbital angular momentum is not conserved in general.

## Why the Ordinary Kepler Argument Fails

In the ordinary Kepler problem,

$$
M\ddot q=F,
$$

and the gravitational force is central:

$$
F=-\frac{kq}{|q|^3}.
$$

Because $F$ is parallel to $q$,

$$
q\times F=0,
$$

so the torque about the Sun is zero and orbital angular momentum is conserved.

The rolling-disk system is different because the velocity must satisfy

$$
\dot q=ve_\theta.
$$

The acceleration is

$$
\ddot q
=
\dot v e_\theta
+
v\omega e_\theta^\perp,
$$

where

$$
e_\theta^\perp=(-\sin\theta,\cos\theta).
$$

The perpendicular part of the acceleration is associated with the constraint force needed to enforce the rolling condition. This constraint force is not generally radial, so it can exert torque about the Sun.

A direct derivative also shows that conservation fails in general. Starting from

$$
L_{\mathrm{orb}}
=
Mv(x\sin\theta-y\cos\theta),
$$

we obtain

$$
\frac{dL_{\mathrm{orb}}}{dt}
=
(x\cos\theta+y\sin\theta)
\left[
Mv\omega
-
\frac{k(x\sin\theta-y\cos\theta)}{(x^2+y^2)^{3/2}}
\right].
$$

This expression is not generally zero.

## Special Case 2

Case 2 is a special circular solution. Its maximum angular-momentum change is only about $1.48\times10^{-10}$, which is numerical error.

For this case,

$$
q\cdot e_\theta=0,
$$

and the exact circular kinematic relation

$$
\omega=\frac{v}{r}
$$

holds. Thus the orbital angular momentum happens to remain constant for this special trajectory even though it is not a conserved quantity for the system in general.

---

# 6. Heading Angular Momentum

The heading angular momentum is

$$
L_\theta=J\omega.
$$

Since

$$
\dot\omega=0,
$$

and $J$ is constant,

$$
\frac{dL_\theta}{dt}
=
J\dot\omega
=0.
$$

Therefore $L_\theta$ should be conserved exactly in this simplified model.

## Numerical Results

| Case | Initial $L_\theta$ | Minimum | Maximum | Maximum Absolute Change |
| --- | ---: | ---: | ---: | ---: |
| Case 1 | 0.05 | 0.05 | 0.05 | 0.0 |
| Case 2 | 0.10 | 0.10 | 0.10 | 0.0 |
| Case 3 | 0.05 | 0.05 | 0.05 | 0.0 |
| Case 4 | 0.05 | 0.05 | 0.05 | 0.0 |

![Heading angular momentum](result%20images/heading_angular_momentum_all_cases.png)

The change from the initial value is shown below.

![Change in heading angular momentum](result%20images/heading_angular_momentum_change.png)

The numerical change is exactly zero for all four simulations. This agrees with the analytical equation $\dot\omega=0$.

The final conservation picture is:

- Energy is conserved.
- Ordinary orbital angular momentum is not conserved in general.
- Heading angular momentum is conserved.

---

# 7. LLM Prediction and Comparison with the Numerical Results

The LLM was asked to predict the conserved quantities before interpreting the numerical results.

## Representative Prompt

> Consider the flat-Earth nonholonomic Kepler model with the first-order equations
>
> $$
> \dot{x}=v\cos\theta,
> $$
>
> $$
> \dot{y}=v\sin\theta,
> $$
>
> $$
> \dot{v}
> =
> -\frac{k}{M}\frac{q\cdot e_\theta}{|q|^3},
> $$
>
> $$
> \dot\theta=\omega,
> $$
>
> $$
> \dot\omega=0.
> $$
>
> Predict whether total energy, ordinary orbital angular momentum, and heading angular momentum should be conserved. Explain the prediction and distinguish the rolling constraint from the ordinary Kepler problem.

## LLM Prediction

The LLM predicted:

- total energy should be conserved;
- ordinary orbital angular momentum should not be conserved in general;
- heading angular momentum should be conserved.

The reasoning was that the nonholonomic rolling constraint changes the usual central-force argument. Gravity is radial, but the constraint introduces an additional force needed to keep the center velocity aligned with the heading. That force can generate a torque about the Sun.

The heading momentum prediction was simpler: since $\dot\omega=0$, the quantity $J\omega$ must remain constant.

## Comparison

| Quantity | LLM Prediction | Numerical Result |
| --- | --- | --- |
| Energy $E$ | Conserved | Conserved to approximately $10^{-9}$ numerical error |
| Orbital angular momentum $L_{\mathrm{orb}}$ | Not conserved in general | Large variation in Cases 1, 3, and 4 |
| Heading angular momentum $L_\theta$ | Conserved | Exactly constant in all four simulations |

The prediction was consistent with the numerical experiments.

The full LLM comparison is stored in:

`code/question_LLM_step8.ipynb`

---

# 8. Discussion Questions

## 8.1 Why is this system not the ordinary Kepler problem?

The ordinary Kepler problem describes an unconstrained point mass moving under a central inverse-square force:

$$
M\ddot q
=
-\frac{kq}{|q|^3}.
$$

The acceleration is determined directly by gravity, and the velocity can point in any direction allowed by the dynamics.

The rolling-disk problem has an additional nonholonomic constraint:

$$
\dot q=ve_\theta.
$$

The center of the disk can only move along its current heading. The heading itself evolves according to

$$
\dot\theta=\omega.
$$

Therefore gravity cannot simply accelerate the disk directly toward the Sun. Only the component of gravity along the heading changes $v$, while the rolling constraint supplies whatever additional force is needed to maintain the permitted motion.

The model therefore has different equations, different conserved quantities, and very different trajectories from the ordinary Kepler problem.

## 8.2 Why is angular momentum conservation more delicate with a nonholonomic constraint?

For an unconstrained particle under a central force, the torque is

$$
q\times F=0,
$$

which immediately implies conservation of orbital angular momentum.

For the rolling disk, the gravitational force is still central, but it is not the only effective force acting on the center. A constraint force must enforce the rolling condition. This force is not generally radial and can therefore exert a torque about the origin.

The numerical experiment demonstrates the result directly. Cases 1, 3, and 4 change their ordinary orbital angular momentum by about 41.9%, 125.9%, and 70.6%, respectively.

Thus the statement "central gravity implies conserved orbital angular momentum" is no longer sufficient when a nonholonomic constraint is present.

## 8.3 What happens if $\omega=0$?

If

$$
\omega=0,
$$

then

$$
\theta(t)=\theta_0
$$

is constant. Therefore the disk always moves along one fixed straight-line direction:

$$
\dot q=ve_{\theta_0}.
$$

The problem reduces to one-dimensional motion along that fixed line in the potential

$$
U=-\frac{k}{|q|}.
$$

The outcome depends on the initial line and energy.

If the fixed line passes through the Sun and the disk moves toward the origin, the disk can fall into the singularity.

If the line misses the Sun, the disk has a nonzero closest distance from the origin and cannot hit the Sun. Depending on the energy, it can pass through the point of closest approach and continue outward, or it can turn around and remain bounded along that line.

Therefore $\omega=0$ does not have one universal outcome. It can lead to infall, passage, escape, or bounded back-and-forth motion depending on the initial conditions.

This case was not one of the four numerical runs above, so this conclusion follows from the reduced equations rather than from a plotted $\omega=0$ simulation.

## 8.4 What happens if $\omega\neq0$? Can bounded or quasi-periodic-looking trajectories form?

Yes. The simulations provide direct examples.

Cases 1, 3, and 4 show repeated looping patterns around the Sun, while Case 2 gives a nearly exact circular trajectory. Over the simulated interval $0\le t\le100$, these trajectories are bounded and have periodic or quasi-periodic-looking structure.

Because

$$
\theta(t)=\theta_0+\omega t,
$$

the allowed direction of motion rotates continuously. Gravity repeatedly projects onto this rotating heading, producing complex looping paths rather than ordinary ellipses.

The finite-time numerical plots show bounded or quasi-periodic-looking behavior, but they do not by themselves prove that every such trajectory remains bounded for all time.

## 8.5 Did the LLM correctly distinguish between holonomic and nonholonomic constraints?

Yes, in the final derivation the LLM correctly treated the rolling condition as a nonholonomic velocity constraint.

The constraint can be written as

$$
-\dot x\sin\theta+\dot y\cos\theta=0.
$$

This restricts the allowed velocities and is not simply a fixed equation of the form

$$
f(x,y,\theta,t)=0
$$

that confines the configuration to a particular curve or surface. That is the key distinction from a holonomic positional constraint.

One thing the LLM got right was recognizing that the ordinary central-force angular-momentum argument could fail because the rolling constraint introduces an additional constraint force. This prediction was verified numerically.

A point that required care was avoiding an overly broad statement such as "nonholonomic systems never conserve angular momentum." Case 2 shows why that statement would be wrong. Its ordinary orbital angular momentum is constant because it is a special circular solution, even though $L_{\mathrm{orb}}$ is not a conserved quantity for general trajectories.

Another limitation is that the LLM could describe the looping trajectories as "bounded" from the figures, but the simulation only runs to $t=100$. The correct conclusion is that they are bounded over the observed interval and look quasi-periodic; long-time boundedness would require additional analysis.

---

# 9. Evaluation of the LLM as a Mechanics Assistant

The LLM was useful in several ways. It organized the problem into the rolling constraint, force projection, effective mass, heading dynamics, conserved quantities, and numerical tests. It also supplied useful mathematical checks instead of only producing Python code. In particular, differentiating the proposed energy and angular momentum formulas made it possible to verify the conservation predictions analytically before relying on the simulation.

The prediction about the three quantities was successful. Energy was predicted to be conserved and was conserved to roughly $10^{-9}$ numerical error. Heading angular momentum was predicted to be conserved and remained exactly constant. Ordinary orbital angular momentum was predicted not to be conserved in general, and three of the four trajectories showed large variations.

The multiple initial conditions were important for evaluating the LLM's claim. If only Case 2 had been simulated, the nearly perfect circular orbit would have suggested that ordinary orbital angular momentum was conserved. Cases 1, 3, and 4 showed that this was only a special case. This is a good example of why an LLM suggestion should be tested across multiple conditions rather than accepted from one favorable example.

The LLM also needed some interpretation rather than blind acceptance. Statements such as "the constraint destroys angular momentum" are useful shorthand but are too strong if taken literally. The more precise statement is that ordinary orbital angular momentum is not a general conserved quantity because the constraint force can exert torque about the origin. Special trajectories can still have constant orbital angular momentum.

Overall, the LLM was useful as a derivation and diagnostic assistant, but the most important conclusions came from combining its suggestions with mathematical checks and numerical experiments. The project supports using an LLM to propose mechanics arguments, provided that each proposed conservation law is verified independently.

---

# 10. Final Summary

The flat-Earth rolling-disk model is not an ordinary Kepler problem because the disk is subject to the nonholonomic velocity constraint

$$
\dot q=ve_\theta.
$$

The numerical experiments produced circular and looping trajectories that differ strongly from ordinary Kepler ellipses.

The conservation tests gave the following result:

| Quantity | Result |
| --- | --- |
| Total energy $E$ | Conserved to numerical error |
| Ordinary orbital angular momentum $L_{\mathrm{orb}}$ | Not conserved in general |
| Heading angular momentum $L_\theta$ | Conserved exactly |

The energy errors were at the level of about $10^{-9}$ or smaller. Ordinary orbital angular momentum changed substantially for Cases 1, 3, and 4, while Case 2 was a special circular solution with constant orbital angular momentum. Heading angular momentum remained exactly constant because $\dot\omega=0$.

The central lesson is that a central gravitational force does not automatically give the ordinary Kepler conservation laws when the motion is subject to a nonholonomic rolling constraint. The constraint changes the admissible motion and introduces forces that can exert torque about the origin.

The LLM correctly predicted the main conservation behavior, but the predictions became trustworthy only after analytical and numerical verification.

---

# Files Submitted with This Report

## Code

- `code/derivation(step1).ipynb`
- `code/derivation(step2).ipynb`
- `code/(step3-7).ipynb`
- `code/question_LLM_step8.ipynb`

## Data

- `data/case_1_simulation.csv`
- `data/case_2_simulation.csv`
- `data/case_3_simulation.csv`
- `data/case_4_simulation.csv`
- `data/energy_conservation_summary.csv`
- `data/orbital_angular_momentum_summary.csv`
- `data/heading_angular_momentum_summary.csv`

## Figures

All generated trajectory and conservation figures are stored in the `result images` directory.
